# El codigo, pieza por pieza

Recorrido completo del paquete `abs_affinity_based_slotting`, de a una pieza por vez, corriendo
sobre el deposito `small` de v2. Cada seccion sigue el mismo esquema: que es la pieza, que
recibe, que devuelve, que valores admite, y una celda que la muestra funcionando.

## Como esta organizado el paquete

El paquete son ocho subpaquetes. Cada uno depende solo de los anteriores, nunca de los que
siguen. Leidos en orden son tambien el recorrido de este notebook.

| subpaquete | responde a | produce |
|---|---|---|
| `data/` | como leo las tablas y como las parto en train y test | cinco DataFrames; train, test y fecha de corte |
| `demand/` | cuanto se pide cada producto y cuales se piden juntos | demanda `f`; co-ocurrencia; afinidad `A` |
| `warehouse/` | que productos hay que ubicar y que tan lejos esta cada lugar | universo; costo de acceso `c`; distancias `D` |
| `slotting/` | como se representa el problema, una solucion y su costo | `SlottingInstance`; `Assignment`; objetivo |
| `clustering/` | como agrupo productos | una etiqueta por producto |
| `methods/` | como resuelvo | un `Assignment` a partir de una instancia |
| `evaluation/` | cuanto camina el picker con un plan dado | distancia por batch sobre test |
| `simulation/` | que pasa cuando el deposito cambia con las reposiciones | lo mismo, pero con el estado evolucionando |

## Dos convenciones que atraviesan todo

**Registries.** Todo lo que se puede intercambiar (metricas de afinidad, filtros, agrupamientos,
metodos, politicas de reposicion) esta anotado en un diccionario `nombre -> clase` llamado
*registry*. Se pide la clase por nombre y se la instancia con sus parametros:

```python
builder = affinity_registry.get("jaccard")()        # clase, y despues instancia
filtro  = filter_registry.get("top_k")(k=10)
```

`registry.names()` lista los nombres disponibles. Agregar una variante es escribir una clase y
registrarla; nada de lo que recorre el registry cambia.

**Ids e indices.** Afuera del nucleo todo se identifica con los ids del dato (`SKU-00042`,
`location_id` 181, `A03-07`). Adentro de `SlottingInstance` y de la simulacion se trabaja con
posiciones enteras, porque numpy y scipy son rapidos con enteros y lentos con strings. La
instancia guarda los mapas en las dos direcciones. Las distancias siempre estan en pulgadas.

In [1]:
import inspect
import time

import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", None)


def describir(registry):
    """Una fila por elemento del registry: nombre, clase y parametros del constructor."""
    rows = []
    for name in registry.names():
        cls = registry.get(name)
        if cls.__init__ is object.__init__:
            params = "(ninguno)"
        else:
            sig = inspect.signature(cls.__init__)
            params = ", ".join(str(p) for n, p in sig.parameters.items() if n != "self")
        rows.append({"nombre": name, "clase": cls.__name__, "parametros del constructor": params})
    return pd.DataFrame(rows).set_index("nombre")

## 1. `data/`: leer y partir

### 1.1 Elegir el dataset: `dataset_dir`

`dataset_dir(version, size)` en `config.py` devuelve la carpeta con los cinco parquets.

| llamada | carpeta | que es |
|---|---|---|
| `dataset_dir("v1")` | `data/v1/raw` | el deposito original: 30 dias, 27.000 SKUs, sin `order_id` |
| `dataset_dir("v2", "small")` | `data/v2/small` | 4 pasillos, 1.728 SKUs, 6 meses |
| `dataset_dir("v2", "medium")` | `data/v2/medium` | 8 pasillos, 6.912 SKUs, 6 meses |
| `dataset_dir("v2", "big")` | `data/v2/big` | 16 pasillos, 24.883 SKUs, 6 meses |

Cualquier otro valor lanza `ValueError`; una carpeta que no existe lanza `FileNotFoundError`.

In [2]:
from abs_affinity_based_slotting.config import dataset_dir, V2_SIZES, DOCK

print("tamanos de v2:", V2_SIZES)
print("dock se llama:", DOCK)

SIZE = "small"
DATA_DIR = dataset_dir("v2", SIZE)
print("carpeta:", DATA_DIR)

tamanos de v2: ('small', 'medium', 'big')
dock se llama: DOCK
carpeta: /home/santiago.fada/personales/ABS-Affinity-based-slotting/data/v2/small


### 1.2 Leer las tablas: `WarehouseDataLoader`

`WarehouseDataLoader(carpeta).load_all()` lee los cinco parquets y devuelve un
`WarehouseDataset`, que es un objeto con cinco atributos, uno por tabla. No limpia ni
transforma nada; es solo lectura.

| atributo | una fila por | columnas |
|---|---|---|
| `coordinates` | bay, mas el dock | `bay_id`, `zone`, `aisle`, `bay_number`, `side`, `x`, `y` |
| `distances` | par de bays no ordenado | `bay_a`, `bay_b`, `distance_in` |
| `initial_stock` | ubicacion exacta | `location_id`, `location_name`, `bay_id`, `sku`, `merchant_account_id`, `units` |
| `picking_events` | linea de pick | `batch_id`, `order_id` (solo v2), `timestamp`, `user_id`, `location_id`, `location_name`, `bay_name`, `sku`, `merchant_account_id`, `quantity` |
| `replenishment_events` | reposicion | `timestamp`, `user_id`, `source_location_id`, `target_location_id`, `location_name`, `sku`, `merchant_account_id`, `quantity` |

In [3]:
from abs_affinity_based_slotting.data import WarehouseDataLoader

ds = WarehouseDataLoader(DATA_DIR).load_all()
for name in ("coordinates", "distances", "initial_stock", "picking_events", "replenishment_events"):
    table = getattr(ds, name)
    print(f"{name:22s} {str(table.shape):>14s}   {list(table.columns)}")

coordinates                   (81, 7)   ['bay_id', 'zone', 'aisle', 'bay_number', 'side', 'x', 'y']
distances                   (3240, 3)   ['bay_a', 'bay_b', 'distance_in']
initial_stock               (1920, 6)   ['location_id', 'location_name', 'bay_id', 'sku', 'merchant_account_id', 'units']
picking_events            (95043, 10)   ['batch_id', 'order_id', 'timestamp', 'user_id', 'location_id', 'location_name', 'bay_name', 'sku', 'merchant_account_id', 'quantity']
replenishment_events        (9417, 8)   ['timestamp', 'user_id', 'source_location_id', 'target_location_id', 'location_name', 'sku', 'merchant_account_id', 'quantity']


### 1.3 Partir en train y test: `split_picking_events`

`split_picking_events(picking_events, test_size=0.2)` devuelve un `TemporalSplit` con tres
atributos: `train`, `test` (dos DataFrames con las mismas columnas que `picking_events`) y
`cutoff` (el primer timestamp que pertenece a test).

El corte es **temporal y por batch**: cada batch se ordena por su primera linea y el 20% mas
reciente va a test, entero. Nunca se parte un batch, porque el batch es la unidad sobre la que
se mide el recorrido. En v2 ninguna orden esta repartida en dos batches, asi que tampoco se
parte ninguna orden.

Todo lo que se aprende (demanda, afinidad) sale de `train`. Todo lo que se mide sale de `test`.

In [4]:
from abs_affinity_based_slotting.data import split_picking_events

split = split_picking_events(ds.picking_events, test_size=0.2)
print("corte:", split.cutoff)
for name, part in (("train", split.train), ("test", split.test)):
    print(f"{name:5s} {len(part):>8,} lineas | {part['batch_id'].nunique():>5,} batches | {part['order_id'].nunique():>7,} ordenes | {part['sku'].nunique():>5,} SKUs")

corte: 2026-05-25 16:00:22.733333
train   75,787 lineas |   868 batches |  21,700 ordenes | 1,725 SKUs
test    19,256 lineas |   218 batches |   5,450 ordenes | 1,518 SKUs


## 2. `demand/`: que se pide y que se pide junto

### 2.1 La demanda de cada producto: `build_sku_demand`

`build_sku_demand(picking_train)` agrega las lineas de pick por SKU y devuelve un DataFrame
con una fila por SKU, ordenado de mas a menos pedido.

| columna | que es |
|---|---|
| `sku` | el producto |
| `merchant_account_id` | su merchant |
| `pick_lines` | cuantas lineas de pick tuvo, o sea cuantas veces alguien fue hasta su ubicacion |
| `total_units` | cuantas unidades se llevaron en total |
| `unique_batches` | en cuantos batches distintos aparecio |
| `first_pick_date`, `last_pick_date` | primer y ultimo pick |

La demanda `f` que usa el modelo es `pick_lines`, no las unidades: lo que cuesta es el viaje
hasta la ubicacion, y ese viaje es uno por linea sin importar cuantas unidades se lleven.
Solo aparecen los SKUs que se pidieron en train; los demas entran despues con demanda cero.

In [5]:
from abs_affinity_based_slotting.demand import build_sku_demand

sku_demand = build_sku_demand(split.train)
print("SKUs con demanda en train:", len(sku_demand))
sku_demand.head()

SKUs con demanda en train: 1725


,sku,merchant_account_id,pick_lines,total_units,unique_batches,first_pick_date,last_pick_date
0,SKU-00839,MER-004,3196,6427,855,2026-01-01 00:09:47.333333,2026-05-25 12:09:24.933333
1,SKU-01025,MER-003,3191,6259,850,2026-01-01 00:13:13.933333,2026-05-25 12:13:10.733333
2,SKU-01069,MER-001,3183,6357,852,2026-01-01 00:14:15.533333,2026-05-25 12:02:09.333333
3,SKU-00022,MER-002,3099,6191,852,2026-01-01 00:00:22.733333,2026-05-25 12:21:43.933333
4,SKU-00419,MER-003,1839,3700,782,2026-01-01 00:04:54.133333,2026-05-25 12:01:07.733333


### 2.2 Que se pide junto: `build_cooccurrence`

`build_cooccurrence(picking, skus=None, batch_col="batch_id", sku_col="sku")` cuenta, para cada
par de SKUs, en cuantas *unidades de agrupacion* aparecen los dos. Devuelve un objeto
`Cooccurrence` con cuatro atributos.

| atributo | que es |
|---|---|
| `skus` | el universo de SKUs, en el orden que define las filas y columnas |
| `matrix` | matriz dispersa `n_ij`: en cuantas unidades aparecen `i` y `j` juntos; diagonal en cero |
| `support` | vector `s_i`: en cuantas unidades aparece `i` |
| `n_batches` | `N`, cuantas unidades de agrupacion hay en total |

La **unidad de agrupacion** la define `batch_col`, y es la decision importante:

| `batch_col` | unidad | disponible en |
|---|---|---|
| `"batch_id"` | el viaje de picking: 25 ordenes de varios merchants, 60 a 90 SKUs | v1 y v2 |
| `"order_id"` | la orden del cliente: 2 a 5 SKUs de un solo merchant | solo v2 |

Si se pasa `skus`, ese es el universo y define el orden de la matriz; los SKUs del universo
que no aparecen en train quedan con soporte cero. Conviene pasar siempre el universo completo
para que la matriz quede alineada con la instancia que se arma despues.

In [6]:
from abs_affinity_based_slotting.demand import build_cooccurrence
from abs_affinity_based_slotting.warehouse import occupied_locations

universe = occupied_locations(ds.initial_stock)["sku"].to_numpy()   # todos los SKUs del stock inicial
print("universo:", len(universe), "SKUs")

co_batch = build_cooccurrence(split.train, skus=universe, batch_col="batch_id")
co_order = build_cooccurrence(split.train, skus=universe, batch_col="order_id")

for name, co in (("por batch", co_batch), ("por orden", co_order)):
    print(f"{name}: N={co.n_batches:>6,} | pares con n_ij>0: {co.matrix.nnz // 2:>8,} | soporte medio: {co.support.mean():.1f}")

# el par que mas veces aparece junto en una orden, contado de las dos maneras
i, j = np.unravel_index(np.argmax(co_order.matrix.toarray()), co_order.matrix.shape)
print(f"\nSKUs {co_order.skus[i]} y {co_order.skus[j]} (el par mas frecuente por orden):")
print(f"  por batch: s_i={co_batch.support[i]}, s_j={co_batch.support[j]}, n_ij={co_batch.matrix[i, j]}")
print(f"  por orden: s_i={co_order.support[i]}, s_j={co_order.support[j]}, n_ij={co_order.matrix[i, j]}")

universo: 1728 SKUs


por batch: N=   868 | pares con n_ij>0:  385,263 | soporte medio: 31.7
por orden: N=21,700 | pares con n_ij>0:   29,421 | soporte medio: 43.9

SKUs SKU-00625 y SKU-01069 (el par mas frecuente por orden):
  por batch: s_i=780, s_j=852, n_ij=771
  por orden: s_i=1834, s_j=3183, n_ij=1063


### 2.3 Convertir conteos en afinidad: `affinity_registry`

Un *builder* de afinidad convierte los tres ingredientes anteriores en un puntaje `a_ij`.
Todos cumplen el mismo contrato (`AffinityBuilder`):

```python
builder.build(cooccurrence.matrix, cooccurrence.support, cooccurrence.n_batches) -> csr_matrix
```

y devuelven una matriz dispersa del mismo tamano, simetrica. Los tres registrados:

| nombre | formula | rango | que premia |
|---|---|---|---|
| `cooccurrence` | `n_ij` | 0 a min(s_i, s_j) | pares populares que coinciden mucho |
| `jaccard` | `n_ij / (s_i + s_j - n_ij)` | 0 a 1 | pares que casi siempre van juntos |
| `cosine` | `n_ij / sqrt(s_i * s_j)` | 0 a 1 | parecido a Jaccard, menos severo si los soportes difieren |

Ninguno tiene parametros. Lift, confianza y PMI estan documentados como candidatos en
`demand/affinity.py` pero no implementados.

In [7]:
from abs_affinity_based_slotting.demand import affinity_registry

describir(affinity_registry)

,clase,parametros del constructor
nombre,,
cooccurrence,CooccurrenceAffinity,(ninguno)
cosine,CosineAffinity,(ninguno)
jaccard,JaccardAffinity,(ninguno)


In [8]:
# el mismo par bajo las tres metricas, con la co-ocurrencia por orden
for name in affinity_registry.names():
    A_name = affinity_registry.get(name)().build(co_order.matrix, co_order.support, co_order.n_batches)
    print(f"{name:13s} a_ij={A_name[i, j]:8.3f} | entradas no nulas: {A_name.nnz:,} | max: {A_name.max():.3f}")

cooccurrence  a_ij=1063.000 | entradas no nulas: 58,842 | max: 1063.000
cosine        a_ij=   0.440 | entradas no nulas: 58,842 | max: 0.447
jaccard       a_ij=   0.269 | entradas no nulas: 58,842 | max: 0.269


### 2.4 Quedarse con los vinculos fuertes: `filter_registry`

La matriz de afinidad completa tiene una entrada por cada par que co-ocurrio alguna vez. Un
filtro la vuelve mas rala. Contrato (`AffinityFilter`):

```python
filtro.filter(A) -> csr_matrix      # mismo tamano, menos entradas, simetrica
```

| nombre | parametro | que conserva |
|---|---|---|
| `top_k` | `k` | los `k` vecinos mas fuertes de cada SKU; una arista sobrevive si esta en el top-k de **alguno** de los dos (union) |
| `mutual_top_k` | `k` | una arista sobrevive solo si esta en el top-k de **ambos** (interseccion); mas estricto |
| `threshold` | `threshold` | las entradas con `a_ij >= threshold` |

La simetria no es opcional: la instancia la exige, porque el calculo incremental del objetivo
(seccion 4.3) la da por supuesta.

In [9]:
from abs_affinity_based_slotting.demand import filter_registry

describir(filter_registry)

,clase,parametros del constructor
nombre,,
mutual_top_k,MutualTopKFilter,k: 'int'
threshold,ThresholdFilter,threshold: 'float'
top_k,TopKFilter,k: 'int'


In [10]:
A_full = affinity_registry.get("jaccard")().build(co_order.matrix, co_order.support, co_order.n_batches)
print(f"jaccard sin filtrar: {A_full.nnz:,} entradas")

for name, kwargs in (("top_k", {"k": 10}), ("top_k", {"k": 5}), ("mutual_top_k", {"k": 10}), ("threshold", {"threshold": 0.05})):
    A_f = filter_registry.get(name)(**kwargs).filter(A_full)
    simetrica = abs(A_f - A_f.T).nnz == 0
    print(f"{name:13s} {str(kwargs):20s} -> {A_f.nnz:>7,} entradas | simetrica: {simetrica}")

A = filter_registry.get("top_k")(k=10).filter(A_full)   # la que se usa en el resto del notebook

jaccard sin filtrar: 58,842 entradas
top_k         {'k': 10}            ->  23,194 entradas | simetrica: True
top_k         {'k': 5}             ->  11,752 entradas | simetrica: True
mutual_top_k  {'k': 10}            ->   9,910 entradas | simetrica: True
threshold     {'threshold': 0.05}  ->   2,370 entradas | simetrica: True


## 3. `warehouse/`: que se ubica y que tan lejos queda cada lugar

Tres funciones, ninguna depende del historico de picking.

| funcion | recibe | devuelve |
|---|---|---|
| `occupied_locations(initial_stock)` | el stock inicial | las filas con SKU; de ahi sale el **universo** de SKUs a ubicar |
| `build_location_costs(initial_stock, distances)` | stock y distancias | una fila por ubicacion con su **costo de acceso** `c`: la distancia de su bay al dock |
| `build_bay_distance_matrix(distances)` | la tabla de pares | la matriz **`D`** bay por bay, simetrica, diagonal cero, indexada por `bay_id` |

`build_location_costs` devuelve las columnas `location_id`, `location_name`, `bay_id`,
`distance_to_dock_in` y `distance_to_dock_m`. El modelo usa la de pulgadas. Dos ubicaciones del
mismo bay tienen el mismo costo, porque no se modela el movimiento dentro del bay.

In [11]:
from abs_affinity_based_slotting.warehouse import build_location_costs, build_bay_distance_matrix, distance_to_dock

loc_costs = build_location_costs(ds.initial_stock, ds.distances)
bay_dist = build_bay_distance_matrix(ds.distances)

print("ubicaciones con costo:", len(loc_costs), "| columnas:", list(loc_costs.columns))
print("D:", bay_dist.shape, "| simetrica:", bool(np.allclose(bay_dist.values, bay_dist.values.T)), "| diagonal cero:", bool((np.diag(bay_dist.values) == 0).all()))
print("distancia al dock: min", distance_to_dock(ds.distances).drop(DOCK).min(), "| max", distance_to_dock(ds.distances).max())
loc_costs.head(3)

ubicaciones con costo: 1920 | columnas: ['location_id', 'location_name', 'bay_id', 'distance_to_dock_in', 'distance_to_dock_m']
D: (81, 81) | simetrica: True | diagonal cero: True
distancia al dock: min 464.0 | max 2024.0


,location_id,location_name,bay_id,distance_to_dock_in,distance_to_dock_m
0,1,A01-01-A-01,A01-01,464.0,11.785594
1,2,A01-01-A-02,A01-01,464.0,11.785594
2,3,A01-01-A-03,A01-01,464.0,11.785594


## 4. `slotting/`: el problema, la solucion y el costo

### 4.1 El problema: `SlottingInstance`

Es la frontera entre pandas y el nucleo numerico. Junta todo lo anterior en un objeto
inmutable que trabaja con indices enteros.

| atributo | forma | que es |
|---|---|---|
| `sku_ids` | `(n,)` | ids de los SKUs; la posicion `i` es el SKU `i` |
| `location_ids` | `(m,)` | ids de las ubicaciones; la posicion `l` es la ubicacion `l` |
| `bay_ids` | `(b,)` | ids de los bays, incluido el dock |
| `demand` | `(n,)` | `f_i`, la demanda de cada SKU |
| `location_cost` | `(m,)` | `c_l`, el costo de acceso de cada ubicacion |
| `location_bay` | `(m,)` | indice del bay de cada ubicacion |
| `bay_distance` | `(b, b)` | `D`, distancia entre bays |
| `affinity` | `(n, n)` dispersa | `A`, la afinidad ya filtrada |
| `merchant_ids` | `(n,)` | merchant de cada SKU, opcional |

Al construirse valida: ids sin duplicados, formas consistentes, sin NaN, valores no negativos,
afinidad simetrica y al menos tantas ubicaciones como SKUs. Si algo falla, lanza `ValueError`
en el momento, no despues. Para pasar de ids a indices y volver tiene `sku_index(id)`,
`location_index(id)`, `bay_index(id)` y sus inversos `sku_id(i)`, `location_id(l)`, `bay_id(b)`.

Se construye de dos maneras:

- `build_instance(sku_demand, location_costs, bay_distance, initial_stock=, skus=, demand_metric="pick_lines", cost_col="distance_to_dock_in", affinity=)`
  arma la instancia a partir de las piezas ya calculadas. `skus` es el universo; los que no
  estan en `sku_demand` entran con demanda cero. Si se pasa `initial_stock`, de ahi sale el
  merchant de cada SKU.
- `build_full_instance(picking_train, initial_stock, distances, affinity_metric="jaccard", cooccurrence_col="batch_id", affinity_filter=None)`
  es el atajo que hace todo lo anterior en una llamada, con el universo tomado del stock
  inicial.

In [12]:
from abs_affinity_based_slotting.slotting import build_instance, build_full_instance, SlottingInstance
from abs_affinity_based_slotting.demand import TopKFilter

instance = build_instance(sku_demand, loc_costs, bay_dist,
                          initial_stock=ds.initial_stock, skus=universe, affinity=A)
print(instance)
print("SKUs con demanda cero:", int((instance.demand == 0).sum()), "| merchants distintos:", len(set(instance.merchant_ids)))

# el atajo equivalente
shortcut = build_full_instance(split.train, ds.initial_stock, ds.distances,
                               affinity_metric="jaccard", cooccurrence_col="order_id",
                               affinity_filter=TopKFilter(k=10))
print("build_full_instance da la misma afinidad:", abs(shortcut.affinity - instance.affinity).nnz == 0)

# ids <-> indices
i = instance.sku_index("SKU-00001")
print(f"\nSKU-00001 es el indice {i}; su demanda es {instance.demand[i]:.0f} y su merchant {instance.merchant_ids[i]}")
l = instance.location_index(1)
print(f"location_id 1 es el indice {l}; esta en el bay {instance.bay_id(instance.bay_of_location_index(l))} a {instance.cost_of_location_index(l):.0f} pulgadas del dock")

SlottingInstance(n_skus=1728, n_locations=1920, n_bays=81, affinity_edges=23194)
SKUs con demanda cero: 3 | merchants distintos: 4


build_full_instance da la misma afinidad: True

SKU-00001 es el indice 0; su demanda es 9 y su merchant MER-003
location_id 1 es el indice 0; esta en el bay A01-01 a 464 pulgadas del dock


### 4.2 La solucion: `Assignment`

Un `Assignment` es un plan: que ubicacion recibe cada SKU. Es un mapa inyectivo, cada SKU va a
una ubicacion y cada ubicacion recibe a lo sumo un SKU, y trabaja con **ids**, no con indices.
Es el objeto que devuelve cualquier metodo y el que recibe el evaluador.

| operacion | que hace |
|---|---|
| `Assignment({sku: location_id, ...})` | construye a partir de un dict; falla si dos SKUs comparten ubicacion |
| `Assignment.from_frame(df, sku_col="sku", location_col="location_id")` | construye a partir de un DataFrame |
| `location_of(sku)` | la ubicacion del SKU |
| `sku_at(location)` | el SKU de la ubicacion, o `None` si esta vacia |
| `swap(sku_a, sku_b)` | intercambia las ubicaciones de dos SKUs, en el lugar, en tiempo constante |
| `copy()`, `to_dict()`, `to_frame()` | copias y exportacion |
| `len(a)`, `sku in a`, `a.skus`, `a.locations` | tamano y recorridos |

El `swap` en tiempo constante es lo que hace posible la busqueda local: probar cientos de miles
de intercambios sin reconstruir nada.

In [13]:
from abs_affinity_based_slotting.slotting import Assignment

toy = Assignment({"SKU-A": 10, "SKU-B": 20, "SKU-C": 30})
print("ubicacion de SKU-B:", toy.location_of("SKU-B"), "| que hay en 30:", toy.sku_at(30), "| que hay en 99:", toy.sku_at(99))
toy.swap("SKU-A", "SKU-C")
print("tras swap(A, C):", toy.to_dict())
try:
    Assignment({"SKU-A": 5, "SKU-B": 5})
except ValueError as e:
    print("dos SKUs en la misma ubicacion:", e)

ubicacion de SKU-B: 20 | que hay en 30: SKU-C | que hay en 99: None
tras swap(A, C): {'SKU-A': 30, 'SKU-B': 20, 'SKU-C': 10}
dos SKUs en la misma ubicacion: Location 5 is assigned to multiple SKUs: 'SKU-A' and 'SKU-B'.


### 4.3 El costo: `slotting_cost` y `swap_cost_delta`

`slotting_cost(assignment, instance, lam=)` devuelve el objetivo

```
C = lam * L + (1 - lam) * Q
L = suma sobre SKUs de   f_i * c[ubicacion de i]                     (demanda por costo de acceso)
Q = suma sobre pares de  a_ij * d[ubicacion de i, ubicacion de j]    (afinidad por distancia)
```

`lam` va de 0 a 1. Con `lam = 1` solo cuenta la demanda; con `lam = 0` solo la afinidad.

`swap_cost_delta(assignment, instance, sku_a, sku_b, lam=)` devuelve cuanto cambiaria `C` si se
intercambian dos SKUs, sin recalcular todo: mira solo a los vecinos de afinidad de los dos.
Negativo quiere decir que el intercambio mejora. Es `O(k)` con afinidad top-k, contra `O(n)`
del calculo completo, y es lo que vuelve viable la busqueda local sobre miles de SKUs.

Dos cosas para tener presentes:

- `C` es un **surrogate**: se calcula sobre train y guia la busqueda, pero el desempeno se
  mide con otra cosa, la distancia de los recorridos sobre test (seccion 7).
- `L` y `Q` **no estan en la misma escala**: `L` suma picks por pulgadas y `Q` suma Jaccard,
  que es menor que 1, por pulgadas. La celda siguiente lo muestra; la seccion 6.7 vuelve sobre
  esto.

In [14]:
from abs_affinity_based_slotting.slotting import slotting_cost, swap_cost_delta
from abs_affinity_based_slotting.methods import DemandGreedySlotting

plan = DemandGreedySlotting().solve(instance)
L = slotting_cost(plan, instance, lam=1.0)
Q = slotting_cost(plan, instance, lam=0.0)
print(f"plan demand_greedy: L = {L:,.0f} | Q = {Q:,.0f} | L / Q = {L / Q:,.0f}")
for lam in (1.0, 0.5, 0.1, 0.01):
    print(f"  lam={lam:<5} C = {slotting_cost(plan, instance, lam=lam):,.1f}")

# el delta incremental coincide con la diferencia de costos completos
a, b = instance.sku_id(0), instance.sku_id(1)
lam = 0.5
delta = swap_cost_delta(plan, instance, a, b, lam=lam)
swapped = plan.copy(); swapped.swap(a, b)
diff = slotting_cost(swapped, instance, lam=lam) - slotting_cost(plan, instance, lam=lam)
print(f"\nintercambiar {a} y {b}: delta incremental = {delta:,.2f} | diferencia completa = {diff:,.2f}")

plan demand_greedy: L = 47,921,528 | Q = 326,992 | L / Q = 147
  lam=1.0   C = 47,921,528.0
  lam=0.5   C = 24,124,260.1
  lam=0.1   C = 5,086,445.9
  lam=0.01  C = 802,937.7

intercambiar SKU-00001 y SKU-00002: delta incremental = 168.29 | diferencia completa = 168.29


### 4.3b Poner los dos terminos en la misma escala: `normalize=True`

`L` suma 1.700 terminos de picks por pulgadas y `Q` suma 23.000 pares de Jaccard por pulgadas;
las dos sumas no tienen relacion entre si y por eso `lam = 0.5` deja la afinidad sin peso.

`build_instance(..., normalize=True)` y `build_full_instance(..., normalize=True)` dividen la
demanda por su suma y la afinidad por su suma, de modo que las dos pasan a ser proporciones:
`f_i` es la fraccion de los picks que se lleva el SKU `i`, y `a_ij` la fraccion del peso de
afinidad que se lleva el par. Con eso

- `L` es la **distancia media al dock de un pick**, en pulgadas;
- `Q` es la **distancia media entre un par afin**, en pulgadas;

y `lam = 0.5` quiere decir que las dos distancias medias pesan igual. Vale para cualquier
metrica: Jaccard sigue decidiendo que pares pesan mas entre si, la normalizacion le da la
escala. Como todo el codigo es lineal en `f` y en `A`, el objetivo, el delta de intercambio y
la politica `objective_greedy` no cambian. Se aplica sobre la afinidad tal como llega, asi
que el filtro va antes.

In [15]:
instance_norm = build_instance(sku_demand, loc_costs, bay_dist,
                               initial_stock=ds.initial_stock, skus=universe, affinity=A, normalize=True)
print("suma de f:", round(instance_norm.demand.sum(), 6), "| suma de A:", round(instance_norm.affinity.sum(), 6))

plan = DemandGreedySlotting().solve(instance_norm)
rows = []
for name, inst in (("sin normalizar", instance), ("normalizada", instance_norm)):
    Ln = slotting_cost(plan, inst, lam=1.0)
    Qn = slotting_cost(plan, inst, lam=0.0)
    rows.append({"instancia": name, "L": round(Ln, 1), "Q": round(Qn, 1), "L / Q": round(Ln / Qn, 2),
                 "C con lam=0.5": round(slotting_cost(plan, inst, lam=0.5), 1)})
pd.DataFrame(rows).set_index("instancia")

suma de f: 1.0 | suma de A: 1.0


,L,Q,L / Q,C con lam=0.5
instancia,,,,
sin normalizar,47921528.0,326992.3,146.55,24124260.1
normalizada,632.3,627.7,1.01,630.0


Con la instancia normalizada, un pick del plan greedy camina en promedio unas 630 pulgadas
hasta su ubicacion, y dos productos afines quedan en promedio a unas 630 pulgadas entre si. Son
dos numeros del mismo orden, y ahora `lam` los pondera de verdad. El resto del notebook sigue
con la instancia sin normalizar, para que las cifras del exacto (seccion 6.7) muestren el
problema de escala tal como estaba.

### 4.4 Recortar el problema: `restrict_instance`

`restrict_instance(instance, sku_indices, location_indices)` devuelve una instancia nueva con
solo esos SKUs y esas ubicaciones, con la afinidad recortada al subconjunto y la geometria
entera. Es lo que usa el bi-nivel para convertir cada zona en un problema independiente, y lo
que permite darle al resolvedor exacto un pedazo chico (seccion 6.7).

In [16]:
from abs_affinity_based_slotting.slotting import restrict_instance

sub = restrict_instance(instance, np.arange(10), np.arange(12))
print(sub)
print("SKUs:", list(sub.sku_ids[:5]), "... | ubicaciones:", list(sub.location_ids[:5]), "...")

SlottingInstance(n_skus=10, n_locations=12, n_bays=81, affinity_edges=0)
SKUs: ['SKU-00001', 'SKU-00002', 'SKU-00003', 'SKU-00004', 'SKU-00005'] ... | ubicaciones: [1, 2, 3, 4, 5] ...


## 5. `clustering/`: agrupar productos

Una estrategia de agrupamiento recibe una instancia y devuelve **una etiqueta entera por SKU**,
alineada con `instance.sku_ids`. No ubica nada; solo dice que SKUs van juntos. El metodo
bi-nivel toma esas etiquetas como zonas. Contrato (`ClusteringStrategy`):

```python
estrategia.cluster(instance) -> np.ndarray de enteros, forma (n,)
```

| nombre | parametros | grupos | criterio |
|---|---|---|---|
| `merchant` | ninguno | uno por merchant | el vendedor dueno del SKU; requiere `instance.merchant_ids` |
| `demand_class` | `thresholds=(0.8, 0.95)` | `len(thresholds) + 1` | clases A/B/C por demanda acumulada: 0 = A (los pocos que hacen el 80%), 1 = B (el 15% siguiente), 2 = C (la cola) |

Se intento agrupar por componentes conexas de la afinidad y colapso en un unico grupo gigante;
la deteccion de comunidades quedo como pendiente (ver `docs/metodo.md`, seccion 5).

In [17]:
from abs_affinity_based_slotting.clustering import clustering_registry

describir(clustering_registry)

,clase,parametros del constructor
nombre,,
demand_class,DemandClassClustering,"thresholds: 'tuple[float, ...]' = (0.8, 0.95)"
merchant,MerchantClustering,(ninguno)


In [18]:
for name, kwargs in (("merchant", {}), ("demand_class", {}), ("demand_class", {"thresholds": (0.5, 0.8, 0.95)})):
    labels = clustering_registry.get(name)(**kwargs).cluster(instance)
    sizes = pd.Series(labels).value_counts().sort_index()
    print(f"{name:13s} {str(kwargs):32s} -> {len(sizes)} grupos, tamanos {sizes.tolist()}")

merchant      {}                               -> 4 grupos, tamanos [437, 422, 427, 442]
demand_class  {}                               -> 3 grupos, tamanos [271, 715, 742]
demand_class  {'thresholds': (0.5, 0.8, 0.95)} -> 4 grupos, tamanos [35, 236, 715, 742]


## 6. `methods/`: de la instancia al plan

Un metodo recibe una instancia y devuelve un `Assignment` que cubre todos sus SKUs. Contrato
(`SlottingMethod`):

```python
metodo.solve(instance) -> Assignment
```

Seis metodos registrados. La tabla siguiente sale del codigo: nombre, clase y parametros del
constructor con sus valores por defecto.

In [19]:
from abs_affinity_based_slotting.methods import method_registry

describir(method_registry)

,clase,parametros del constructor
nombre,,
bilevel,BiLevelSlotting,"clustering, zone_solver, location_cost=None"
current,CurrentSlotting,initial_stock: 'pd.DataFrame'
demand_greedy,DemandGreedySlotting,(ninguno)
exact_qap,ExactQAPSlotting,"lam: 'float' = 0.5, time_limit: 'float | None' = None, mip_gap: 'float | None' = None, output: 'bool' = False"
linear_assignment,LinearAssignmentSlotting,(ninguno)
swap_search,SwapSearchSlotting,"seed_method=None, lam: 'float' = 0.5, max_passes: 'int' = 50"


Que hace cada uno y que decide:

| nombre | tipo | usa `A` | que hace |
|---|---|---|---|
| `current` | benchmark | no | lee el slotting vigente del stock inicial. Recibe `initial_stock`. Es el numero a superar |
| `demand_greedy` | baseline | no | ordena SKUs por demanda descendente y ubicaciones por costo ascendente, y los empareja. Sin parametros |
| `linear_assignment` | exacto para `lam = 1` | no | resuelve la asignacion lineal optima con el algoritmo hungaro. Sin parametros. O(n^3), sirve hasta unos 5.000 SKUs |
| `swap_search` | busqueda local | si | parte de una semilla (`seed_method`, por defecto `demand_greedy`) y acepta intercambios que bajan `C` con peso `lam`, hasta `max_passes` pasadas o hasta que no haya mejora |
| `bilevel` | descomposicion | si | agrupa con `clustering`, reparte ubicaciones entre grupos resolviendo un problema de transporte, y resuelve cada zona con `zone_solver`, que es cualquier otro metodo. `location_cost` opcional cambia el criterio de reparto (distancia al dock, o posicion en el recorrido para zonas compactas) |
| `exact_qap` | exacto | si | escribe el problema completo para el solver. `lam`, `time_limit` en segundos, `mip_gap` tolerancia relativa, `output` para ver el log. Solo instancias chicas |

`bilevel` y `exact_qap` usan el solver con la licencia del `.env`. Las secciones que siguen
corren cada uno sobre `small`.

### 6.1 `current`: el benchmark

In [20]:
M = lambda name, *args, **kwargs: method_registry.get(name)(*args, **kwargs)
assignments, runtimes = {}, {}

def correr(nombre, metodo, inst=instance):
    t0 = time.time()
    assignments[nombre] = metodo.solve(inst)
    runtimes[nombre] = time.time() - t0
    print(f"{nombre:42s} {runtimes[nombre]:7.2f} s  {assignments[nombre]}")

correr("current", M("current", ds.initial_stock))

current                                       0.04 s  Assignment(n_skus=1728)


### 6.2 `demand_greedy` y `linear_assignment`: solo demanda

Los dos ignoran la afinidad. El greedy deberia coincidir con el optimo lineal, porque para
`lam = 1` ordenar por demanda y por costo es exactamente la solucion optima.

In [21]:
correr("demand_greedy", M("demand_greedy"))
correr("linear_assignment", M("linear_assignment"))
print("\nL greedy   :", f"{slotting_cost(assignments['demand_greedy'], instance, lam=1.0):,.0f}")
print("L hungaro  :", f"{slotting_cost(assignments['linear_assignment'], instance, lam=1.0):,.0f}")

demand_greedy                                 0.00 s  Assignment(n_skus=1728)


linear_assignment                             2.87 s  Assignment(n_skus=1728)

L greedy   : 47,921,528
L hungaro  : 47,921,528


### 6.3 `swap_search`: busqueda local con afinidad

Parte del plan del greedy y prueba, para cada SKU, intercambiarlo con sus vecinos de afinidad.
Acepta el intercambio si `swap_cost_delta` es negativo. Con `lam = 0.5` casi no mueve nada,
por la diferencia de escala entre `L` y `Q`; con `lam` chico si.

In [22]:
for lam in (0.5, 0.01):
    correr(f"swap_search lam={lam}", M("swap_search", lam=lam))
    a = assignments[f"swap_search lam={lam}"]
    print(f"   L = {slotting_cost(a, instance, lam=1.0):,.0f} | Q = {slotting_cost(a, instance, lam=0.0):,.0f}")

swap_search lam=0.5                          11.34 s  Assignment(n_skus=1728)
   L = 47,924,960 | Q = 310,781


swap_search lam=0.01                         14.20 s  Assignment(n_skus=1728)
   L = 48,836,240 | Q = 264,965


### 6.4 `bilevel`: agrupar, repartir zonas, resolver cada zona

Tres decisiones enchufables: con que se agrupa (`clustering`), con que se resuelve cada zona
(`zone_solver`, cualquier metodo del registry) y con que criterio se reparten las ubicaciones
entre grupos (`location_cost`; por defecto la distancia al dock, que produce bandas
concentricas).

In [23]:
correr("bilevel[merchant, zona=demand_greedy]", M("bilevel", clustering_registry.get("merchant")(), M("demand_greedy")))
correr("bilevel[demand_class, zona=demand_greedy]", M("bilevel", clustering_registry.get("demand_class")(), M("demand_greedy")))

Set parameter WLSAccessID


Set parameter WLSSecret


Set parameter LicenseID to value 2805866


Academic license 2805866 - for non-commercial use only - registered to sa___@mi.unc.edu.ar


bilevel[merchant, zona=demand_greedy]         1.10 s  Assignment(n_skus=1728)
Set parameter WLSAccessID


Set parameter WLSSecret


Set parameter LicenseID to value 2805866


Academic license 2805866 - for non-commercial use only - registered to sa___@mi.unc.edu.ar


bilevel[demand_class, zona=demand_greedy]     0.05 s  Assignment(n_skus=1728)


### 6.5 El objetivo de cada plan

`L` y `Q` de cada plan, en la escala de train. Se ve la diferencia de escala entre los dos
terminos y que los metodos por demanda casi no tocan `Q`.

In [24]:
pd.DataFrame({
    name: {"L (lam=1)": slotting_cost(a, instance, lam=1.0), "Q (lam=0)": slotting_cost(a, instance, lam=0.0), "seg": runtimes[name]}
    for name, a in assignments.items()
}).T.round(1)

,L (lam=1),Q (lam=0),seg
current,90271808.0,380322.2,0.0
demand_greedy,47921528.0,326992.3,0.0
linear_assignment,47921528.0,328161.8,2.9
swap_search lam=0.5,47924960.0,310781.4,11.3
swap_search lam=0.01,48836240.0,264964.5,14.2
"bilevel[merchant, zona=demand_greedy]",78289424.0,251162.6,1.1
"bilevel[demand_class, zona=demand_greedy]",47921528.0,328609.9,0.1


### 6.6 `exact_qap`: que modelo escribe y por que no entra la instancia completa

Escribe el problema tal como esta en la formulacion y se lo entrega a un solver de
programacion entera:

- una variable binaria `x_il` por cada par SKU, ubicacion;
- cada SKU en exactamente una ubicacion, cada ubicacion con a lo sumo un SKU;
- objetivo `lam * L + (1 - lam) * Q`, con `Q` recorriendo las aristas de `A` y todos los pares
  de ubicaciones.

Para `small` serian 1.728 por 1.920, unos 3,3 millones de binarias, y el termino cuadratico
tendria una entrada por arista de `A` y par de ubicaciones. Aunque entrara en memoria, el QAP
es NP-hard y los metodos exactos dejan de terminar con unas pocas decenas de objetos.

Su papel es dar el **optimo probado sobre un pedazo chico**, y con eso medir cuanto pierden las
heuristicas. El unico criterio de parada que se usa es `time_limit`: si el solver termina
antes, la solucion es el optimo probado; si llega al limite, devuelve la mejor que encontro.

### 6.7 Armar un pedazo chico y comparar

Para que la afinidad tenga algo que decir, el pedazo se arma como un grupo conectado: se parte
del SKU mas demandado y se agregan sus vecinos de afinidad mas fuertes hasta juntar `n`. Las
ubicaciones son las que esos SKUs ocupan hoy mas cuatro huecos libres, para que haya donde
moverse. Se compara el exacto contra el greedy y la busqueda local para varios `lam`.

In [25]:
from abs_affinity_based_slotting.methods import current_assignment

A_csr = instance.affinity.tocsr()
current = current_assignment(ds.initial_stock, skus=instance.sku_ids)
occupied_idx = {instance.location_index(loc) for loc in current.locations}
free_idx = [l for l in range(instance.n_locations) if l not in occupied_idx]

def pedazo(n_skus, n_free=4):
    chosen = [int(np.argmax(instance.demand))]
    while len(chosen) < n_skus:
        frontier = {}
        for s in chosen:
            row = A_csr.getrow(s)
            for j, w in zip(row.indices, row.data):
                if j not in chosen:
                    frontier[j] = max(frontier.get(j, 0.0), w)
        if not frontier:
            break
        chosen.append(max(frontier, key=frontier.get))
    skus = np.array(chosen)
    locations = [instance.location_index(current.location_of(instance.sku_id(s))) for s in skus]
    return restrict_instance(instance, skus, np.array(locations + free_idx[:n_free]))

small_inst = pedazo(20)
print(small_inst)

SlottingInstance(n_skus=20, n_locations=24, n_bays=81, affinity_edges=256)


In [26]:
TIME_LIMIT = 60
Ls = lambda a: slotting_cost(a, small_inst, lam=1.0)
Qs = lambda a: slotting_cost(a, small_inst, lam=0.0)
greedy_s = M("demand_greedy").solve(small_inst)
print(f"escala en el pedazo: L = {Ls(greedy_s):,.0f} | Q = {Qs(greedy_s):,.0f} | L / Q = {Ls(greedy_s) / Qs(greedy_s):,.0f}")

rows = []
for lam in (1.0, 0.5, 0.01, 0.0):
    t0 = time.time()
    exact = M("exact_qap", lam=lam, time_limit=TIME_LIMIT).solve(small_inst)
    dt = time.time() - t0
    swaps = M("swap_search", lam=lam).solve(small_inst)
    C = lambda a: slotting_cost(a, small_inst, lam=lam)
    rows.append({"lam": lam, "seg exacto": round(dt, 1), "optimo probado": dt < TIME_LIMIT,
                 "C exacto": round(C(exact), 1), "L exacto": round(Ls(exact)), "Q exacto": round(Qs(exact)),
                 "gap greedy": f"{100 * (C(greedy_s) / C(exact) - 1):.2f}%",
                 "gap swaps": f"{100 * (C(swaps) / C(exact) - 1):.2f}%"})
pd.DataFrame(rows).set_index("lam")

escala en el pedazo: L = 7,831,656 | Q = 4,977 | L / Q = 1,574
Set parameter WLSAccessID


Set parameter WLSSecret


Set parameter LicenseID to value 2805866


Academic license 2805866 - for non-commercial use only - registered to sa___@mi.unc.edu.ar


Set parameter WLSAccessID


...

Set parameter WLSAccessID


Set parameter WLSSecret


Set parameter LicenseID to value 2805866


Academic license 2805866 - for non-commercial use only - registered to sa___@mi.unc.edu.ar


,seg exacto,optimo probado,C exacto,L exacto,Q exacto,gap greedy,gap swaps
lam,,,,,,,
1.00,1.6,True,7831656.0,7831656,4893,0.00%,0.00%
0.50,3.0,True,3918258.7,7831656,4861,0.00%,0.00%
0.01,18.9,True,83036.7,7835232,4732,0.25%,0.08%
0.00,61.8,False,4642.6,8081616,4643,7.20%,0.20%


Como leer la tabla:

- `lam = 1`: sin termino cuadratico el problema es una asignacion lineal y el exacto coincide
  con el greedy. Es una verificacion de que el modelo esta bien escrito.
- `lam = 0.5`: el exacto sigue coincidiendo con el greedy, porque `L` pesa unas 1.500 veces mas
  que `Q`. Con este `lam` la afinidad no cuenta, y eso vale tambien para los barridos de v1.
- `lam = 0.01`: la afinidad empieza a mover la solucion; el greedy pierde algo y la busqueda
  local queda muy cerca del optimo.
- `lam = 0`: QAP puro. El solver no lo cierra en el tiempo dado ni con 20 SKUs (`optimo probado
  = False`), asi que el gap es una cota inferior. La busqueda local queda a una fraccion de
  punto; el greedy, que ignora `A`, pierde varios.

Que `lam` solo sea un peso interpretable si `L` y `Q` se normalizan antes es una decision de la
formulacion que esta pendiente (`docs/modelo.md`, seccion 4.7).

In [27]:
# donde esta la pared: tiempo del exacto segun el tamano del pedazo, con lam = 0.01
rows = []
for n in (8, 12, 16, 20, 30):
    si = pedazo(n)
    t0 = time.time()
    M("exact_qap", lam=0.01, time_limit=TIME_LIMIT).solve(si)
    dt = time.time() - t0
    rows.append({"n SKUs": si.n_skus, "m ubicaciones": si.n_locations, "aristas de A": si.affinity.nnz,
                 "seg": round(dt, 1), "optimo probado": dt < TIME_LIMIT})
pd.DataFrame(rows).set_index("n SKUs")

Set parameter WLSAccessID


Set parameter WLSSecret


Set parameter LicenseID to value 2805866


Academic license 2805866 - for non-commercial use only - registered to sa___@mi.unc.edu.ar


Set parameter WLSAccessID


...

Set parameter WLSAccessID


Set parameter WLSSecret


Set parameter LicenseID to value 2805866


Academic license 2805866 - for non-commercial use only - registered to sa___@mi.unc.edu.ar


,m ubicaciones,aristas de A,seg,optimo probado
n SKUs,,,,
8,12,56,0.2,True
12,16,122,2.0,True
16,20,192,4.1,True
20,24,256,18.8,True
30,34,334,64.3,False


Con `lam = 0.01` el exacto prueba optimalidad en segundos hasta unos 20 SKUs y corta por tiempo
a los 30. Una zona por merchant tiene mas de 400 SKUs en `small` y mas de 200 en `big`, asi que
en el bi-nivel el resolvedor de zona es en la practica la busqueda por intercambios, y el
exacto queda como vara sobre pedazos de este tamano.

## 7. `evaluation/`: cuanto camina el picker

### 7.1 El recorrido de un batch

El modelo de recorrido es el mismo en el evaluador y en el simulador. Para un batch se toman
los SKUs pedidos, se busca en que ubicacion estan, se pasa a bays, y las bays se ordenan en
**serpentina**: primero por pasillo, dentro del pasillo por numero de bay. La distancia es
dock, primera bay, ..., ultima bay, dock, sumando `D` tramo a tramo. Varios picks en la misma
bay no agregan distancia. No se usa el orden historico de las lineas; se recalcula segun el plan.

Tres funciones chicas en `evaluation/routes.py` lo hacen:

| funcion | que hace |
|---|---|
| `bay_snake_keys(coordinates, bay_ids)` | una clave por bay, `pasillo * 1000 + numero`, para ordenar |
| `snake_order(bays, keys)` | ordena las bays por su clave |
| `route_distance(bays_ordenadas, D, dock)` | suma dock, bays, dock |

In [28]:
from abs_affinity_based_slotting.evaluation import bay_snake_keys, snake_order, route_distance

keys = bay_snake_keys(ds.coordinates, instance.bay_ids)
dock = instance.bay_index(DOCK)

# el primer batch de test, recorrido con el plan del greedy
batch_id = split.test["batch_id"].iloc[0]
skus = split.test.loc[split.test["batch_id"] == batch_id, "sku"].unique()
plan = assignments["demand_greedy"]
bays = np.unique([instance.location_bay[instance.location_index(plan.location_of(s))] for s in skus])
ordered = snake_order(bays, keys[bays])
print(f"batch {batch_id}: {len(skus)} SKUs distintos en {len(bays)} bays")
print("orden de visita:", [instance.bay_id(b) for b in ordered][:12], "...")
print(f"distancia: {route_distance(ordered, instance.bay_distance, dock):,.0f} pulgadas")

batch BATCH-000869: 61 SKUs distintos en 25 bays
orden de visita: ['A01-01', 'A01-02', 'A01-03', 'A01-04', 'A01-05', 'A01-06', 'A01-07', 'A01-08', 'A01-09', 'A01-10', 'A01-11', 'A01-13'] ...
distancia: 6,472 pulgadas


### 7.2 El evaluador estatico: `Evaluator`

`Evaluator.from_tables(coordinates, distances, locations)` arma la geometria una sola vez
(`locations` es cualquier tabla con `location_id` y `bay_id`, por ejemplo el stock inicial).
Despues `evaluate(assignment, picking_test)` hace el recorrido anterior para cada batch de test
y devuelve un `RouteMetrics`:

| atributo | que es |
|---|---|
| `n_batches` | cuantos batches se midieron |
| `total_distance` | suma de todas las distancias |
| `mean_batch_distance`, `median_batch_distance`, `p95_batch_distance` | resumen por batch |
| `runtime_seconds` | tiempo del metodo, si el que llama lo completa |

Antes de medir verifica que todo SKU pedido en test tenga ubicacion en el plan; si falta alguno
lanza `ValueError` en vez de saltearlo. Se llama **estatico** porque supone que el plan se
mantiene intacto durante todo el test, sin reposiciones ni mudanzas.

In [29]:
from abs_affinity_based_slotting.evaluation import Evaluator

evaluator = Evaluator.from_tables(ds.coordinates, ds.distances, ds.initial_stock)
rows = []
for name, a in assignments.items():
    m = evaluator.evaluate(a, split.test)
    rows.append({"metodo": name, "media/batch (in)": round(m.mean_batch_distance), "mediana": round(m.median_batch_distance),
                 "p95": round(m.p95_batch_distance), "total (m)": round(m.total_distance * 0.0254), "seg": round(runtimes[name], 2)})
res = pd.DataFrame(rows).set_index("metodo")
base = res.loc["current", "media/batch (in)"]
res["vs current"] = ((res["media/batch (in)"] - base) / base * 100).round(1).astype(str) + "%"
res

,media/batch (in),mediana,p95,total (m),seg,vs current
metodo,,,,,,
current,6663,6664,6664,36895,0.04,0.0%
demand_greedy,5832,5872,6472,32291,0.00,-12.5%
linear_assignment,5821,5872,6472,32234,2.87,-12.6%
swap_search lam=0.5,5862,5872,6472,32457,11.34,-12.0%
swap_search lam=0.01,5895,5896,6472,32642,14.20,-11.5%
"bilevel[merchant, zona=demand_greedy]",6110,6088,6664,33835,1.10,-8.3%
"bilevel[demand_class, zona=demand_greedy]",5789,5872,6280,32055,0.05,-13.1%


La media y el p95 casi coinciden en todos los planes: con cuatro pasillos y unos 63 SKUs por
batch, el picker recorre el deposito entero casi siempre, y lo unico que cambia es en que punto
de cada pasillo termina. `small` sirve para entender las piezas, no para comparar metodos; para
eso hacen falta `medium` o `big`.

## 8. `simulation/`: el deposito que cambia

El evaluador estatico supone que el plan se respeta para siempre. En el dato no es asi: cuando
un SKU se queda sin stock se lo repone, y el 45% de las veces se lo muda a otra ubicacion. La
simulacion reproduce esa dinamica. Del dato se replica **todo** salvo una cosa, **a donde van
las unidades repuestas**, y esa decision la toma una politica intercambiable. Ahi es donde el
algoritmo de slotting actua en linea.

### 8.1 El estado: `WarehouseState`

Es la contraparte mutable del `Assignment`: no un plan sino la realidad fisica en un instante,
que SKU hay en cada ubicacion y con cuantas unidades. Trabaja en indices. Reglas:

- una ubicacion aloja a lo sumo un SKU, y una ubicacion con 0 unidades sigue siendo del SKU
  mientras sea la unica que tiene;
- un SKU puede ocupar varias ubicaciones a la vez, que es lo que pasa durante una mudanza;
- no se modela capacidad por ubicacion.

| metodo | que hace |
|---|---|
| `locations_of(sku)`, `sku_at(loc)`, `units_at(loc)`, `total_units(sku)` | consultas |
| `free_pool()` | las ubicaciones libres, con muestreo al azar en tiempo constante |
| `add_units(sku, loc, qty)`, `remove_units(sku, loc, qty)` | entregar y consumir |
| `release_empty_locations(sku)` | liberar las ubicaciones vacias del SKU, dejandole al menos una |
| `check_invariants()` | chequeo completo de consistencia |
| `to_frame(instance)` | exportar con ids |

### 8.2 Los eventos: `split_replenishments` y `build_event_stream`

`split_replenishments(replenishments, picking_train, initial_stock, cutoff)` corta las
reposiciones en el mismo punto que el picking y repara el borde: si una ubicacion quedaria con
stock negativo en train porque su mudanza se registro despues, trae esa reposicion a train.
Devuelve `(train, test)`.

`build_event_stream(picking, replenishments, instance)` convierte las tablas en una lista de
eventos ordenada por tiempo, ya en indices. Dos tipos:

| evento | atributos |
|---|---|
| `PickBatch` | `batch_id`, `timestamp`, `end_timestamp`, `sku_indices`, `quantities`, `line_timestamps`, `location_indices` (las ubicaciones registradas en el dato) |
| `ReplenEvent` | `timestamp`, `sku_idx`, `quantity`, `source_location_idx`, `dataset_target_idx` (a donde lo mando el generador) |

Si un SKU o una ubicacion del dato no esta en la instancia, lanza `ValueError`.

In [30]:
from abs_affinity_based_slotting.simulation import split_replenishments, build_event_stream, PickBatch, ReplenEvent

replen_train, replen_test = split_replenishments(ds.replenishment_events, split.train, ds.initial_stock, split.cutoff)
events_train = build_event_stream(split.train, replen_train, instance)
events_test = build_event_stream(split.test, replen_test, instance)

for name, ev in (("train", events_train), ("test", events_test)):
    print(f"{name}: {sum(isinstance(e, PickBatch) for e in ev):,} batches y {sum(isinstance(e, ReplenEvent) for e in ev):,} reposiciones")
primer = next(e for e in events_test if isinstance(e, PickBatch))
print(f"\nprimer batch de test: {primer.batch_id}, {primer.n_lines} lineas, de {primer.timestamp} a {primer.end_timestamp}")

train: 868 batches y 7,388 reposiciones
test: 218 batches y 2,029 reposiciones

primer batch de test: BATCH-000869, 89 lineas, de 2026-05-25 16:00:22.733333 a 2026-05-25 16:24:31.133333


### 8.3 Los estados iniciales

| funcion | devuelve |
|---|---|
| `state_from_initial_stock(instance, initial_stock)` | el dia cero, tal como esta en la tabla |
| `state_after_history(state, events, instance)` | el estado despues de esos eventos, calculado por aritmetica (inicial mas entregas menos picks); devuelve tambien un `HistorySummary` con `n_batches`, `n_replens`, `n_relocations`, `n_multi_location_skus`, `n_homeless_skus` |
| `reslot_state(state, assignment, instance)` | el estado que resulta de aplicar un plan como re-slotting completo: cada SKU se muda a su ubicacion planificada con las unidades que tenia |

El baseline `current` se simula desde el estado **real en el corte**, no desde el dia cero,
porque durante train el deposito ya cambio. Los planes optimizados se simulan desde
`reslot_state` del estado en el corte.

In [31]:
from abs_affinity_based_slotting.simulation import state_from_initial_stock, state_after_history, reslot_state

state0 = state_from_initial_stock(instance, ds.initial_stock)
state_cut, hist = state_after_history(state0, events_train, instance)
print("dia cero :", state0)
print("al corte :", state_cut)
print(hist)

greedy_plan = assignments["demand_greedy"]
state_greedy = reslot_state(state_cut, greedy_plan, instance)
print("re-slot  :", state_greedy)

dia cero : WarehouseState(n_locations=1920, occupied=1728, free=192, total_units=41992)
al corte : WarehouseState(n_locations=1920, occupied=1728, free=192, total_units=27988)
HistorySummary(n_batches=868, n_replens=7388, n_relocations=3325, n_multi_location_skus=0, n_homeless_skus=0)
re-slot  : WarehouseState(n_locations=1920, occupied=1728, free=192, total_units=27988)


### 8.4 Las politicas de reposicion: `replenishment_policy_registry`

Contrato (`ReplenishmentPolicy`):

```python
politica.choose_destination(evento, estado, contexto) -> indice de ubicacion
```

El destino tiene que ser una ubicacion libre o una que el SKU ya ocupa; el simulador lo
verifica. El `contexto` (`SimulationContext`) trae la instancia, la ubicacion planificada de
cada SKU si hay plan, el `lam` y el generador de numeros al azar de la corrida.

| nombre | parametros | que hace | que modela |
|---|---|---|---|
| `random` | `relocation_probability=0.9` | recarga donde esta; si el stock llego a cero, con esa probabilidad muda a un hueco libre al azar | un deposito sin plan; es como se genero el dato |
| `home` | ninguno | entrega en la ubicacion que fija el plan; si esta ocupada, en el hueco libre mas cercano a ella | un WMS que conoce el plan y lo mantiene |
| `objective_greedy` | ninguno | entrega en la ubicacion de menor costo marginal del objetivo, dado donde esta todo ahora | el algoritmo decidiendo en linea |

In [32]:
from abs_affinity_based_slotting.simulation import replenishment_policy_registry

describir(replenishment_policy_registry)

,clase,parametros del constructor
nombre,,
home,HomePolicy,(ninguno)
objective_greedy,ObjectiveGreedyPolicy,(ninguno)
random,RandomStoragePolicy,relocation_probability: 'float' = 0.9


### 8.5 El simulador: `Simulator`

`Simulator.from_tables(instance, coordinates)` guarda la geometria. Cada llamada a

```python
sim.run(initial_state, events, policy, planned=None, lam=0.5, seed=0) -> SimulationResult
```

copia el estado inicial y recorre los eventos: las lineas de pick consumen stock y anotan la
ubicacion visitada, las reposiciones consultan a la politica, y al cerrar cada batch se mide su
recorrido con el mismo modelo de la seccion 7. `planned` es el plan que se esta simulando; lo
necesitan `home` y `objective_greedy`, y sirve para medir cuantos SKUs quedaron fuera de su
lugar.

| atributo de `SimulationResult` | que es |
|---|---|
| `route_metrics` | un `RouteMetrics`, comparable con el del evaluador estatico |
| `batch_log` | una fila por batch en orden temporal: distancia, lineas, faltantes, fraccion fuera de su lugar |
| `replen_log` | una fila por reposicion: destino, si fue mudanza, si cayo en el lugar planificado |
| `n_relocations`, `shortfall_lines`, `shortfall_units` | mudanzas y faltantes de stock |
| `final_off_home_fraction` | fraccion de SKUs fuera de su lugar planificado al final |
| `final_state` | el `WarehouseState` final |

In [33]:
from abs_affinity_based_slotting.simulation import Simulator

sim = Simulator.from_tables(instance, ds.coordinates)
P = lambda name, *args, **kwargs: replenishment_policy_registry.get(name)(*args, **kwargs)

runs = {
    "current + random": sim.run(state_cut, events_test, P("random"), seed=0),
    "demand_greedy + random": sim.run(state_greedy, events_test, P("random"), planned=greedy_plan, seed=0),
    "demand_greedy + home": sim.run(state_greedy, events_test, P("home"), planned=greedy_plan, seed=0),
    "demand_greedy + objective_greedy lam=1": sim.run(state_greedy, events_test, P("objective_greedy"), planned=greedy_plan, lam=1.0, seed=0),
}
pd.DataFrame({
    name: {"media/batch (in)": round(r.route_metrics.mean_batch_distance), "p95": round(r.route_metrics.p95_batch_distance),
           "mudanzas": r.n_relocations, "faltantes (lineas)": r.shortfall_lines,
           "fuera de su lugar al final": None if np.isnan(r.final_off_home_fraction) else round(r.final_off_home_fraction, 3)}
    for name, r in runs.items()
}).T

,media/batch (in),p95,mudanzas,faltantes (lineas),fuera de su lugar al final
current + random,6662.0,6664.0,693.0,0.0,NaN
demand_greedy + random,6612.0,6664.0,693.0,0.0,0.214
demand_greedy + home,5832.0,6472.0,0.0,0.0,0.000
demand_greedy + objective_greedy lam=1,5832.0,6472.0,0.0,0.0,0.000


Con `home` el resultado coincide con el evaluador estatico (5.832), lo que confirma que el
estatico es el caso particular de la simulacion en que el plan se respeta. Con `random` el
plan optimizado se degrada casi hasta el nivel del slotting vigente en 218 batches: la
diferencia entre las dos filas es lo que vale que la operacion respete el plan.

## 9. Todas las piezas intercambiables, en una tabla

| familia | registry | valores | parametros |
|---|---|---|---|
| dataset | `dataset_dir` | `"v1"`; `"v2"` con `"small"`, `"medium"`, `"big"` | — |
| unidad de co-ocurrencia | `batch_col` / `cooccurrence_col` | `"batch_id"`, `"order_id"` (v2) | — |
| metrica de afinidad | `affinity_registry` | `cooccurrence`, `jaccard`, `cosine` | ninguno |
| filtro de afinidad | `filter_registry` | `top_k`, `mutual_top_k`, `threshold` | `k`; `k`; `threshold` |
| demanda | `demand_metric` en `build_instance` | `pick_lines` (default), `total_units`, `unique_batches` | — |
| agrupamiento | `clustering_registry` | `merchant`, `demand_class` | —; `thresholds` |
| metodo | `method_registry` | `current`, `demand_greedy`, `linear_assignment`, `swap_search`, `bilevel`, `exact_qap` | ver 6 |
| escala de los terminos | `normalize` en `build_instance` | `False` (picks y afinidad crudos), `True` (proporciones; L y Q en pulgadas medias) | — |
| peso | `lam` | 0 a 1; solo comparable con `normalize=True` | — |
| politica de reposicion | `replenishment_policy_registry` | `random`, `home`, `objective_greedy` | `relocation_probability`; —; — |
| estado inicial | funciones | dia cero, al corte, re-slot de un plan | — |
| medicion | `Evaluator` o `Simulator` | estatico, simulado | `test_size` del split; `seed` |

## 10. Pendientes que este recorrido deja a la vista

- Los notebooks de `data1/` estan escritos contra v1 y contra funciones que ya no existen en
  el paquete. Sirven como registro de los experimentos publicados, no para correr.
- La vecindad de `swap_search` es mas chica de lo que dice su docstring: solo prueba
  intercambiar un SKU con sus vecinos de afinidad directos (`estado-del-trabajo.md`, seccion 8).
- `L` y `Q` no estan en la misma escala salvo con `normalize=True` (seccion 4.3b). Todos los
  experimentos de v1 y las corridas de este notebook fuera de esa seccion usan la instancia
  sin normalizar; hay que repetirlos con la normalizada.
- Ni v1 ni v2 muestran afinidad por encima del azar (`docs/modelo.md`, seccion 2.3). Mientras
  eso no cambie, ningun metodo que use `A` puede mostrar aporte.